# 04 — Fragmentation and PagedAttention Simulation

Goal: visualise why naive KV-cache reservation wastes memory and how paged allocation reduces waste.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from vllm_lab.paged_attention_sim import generate_requests, allocation_curve, make_memory_grid

Create a skewed request distribution. This mimics real LLM traffic: many short requests, a few long requests.

In [ ]:
requests = generate_requests(n=200, min_tokens=32, max_tokens=4096, max_reserved_tokens=4096, seed=42)
lengths = [r.actual_tokens for r in requests]
pd.Series(lengths).describe()

In [ ]:
ax = pd.Series(lengths).hist(bins=40)
ax.set_title('Synthetic request length distribution')
ax.set_xlabel('actual tokens')
ax.set_ylabel('request count')

Naive strategy: reserve maximum sequence length for every request. Paged strategy: allocate only the number of fixed-size blocks needed.

In [ ]:
summaries = allocation_curve(requests, block_sizes=[8, 16, 32, 64, 128, 256])
df = pd.DataFrame([s.__dict__ for s in summaries])
df[['strategy', 'block_size', 'total_reserved_tokens', 'total_used_tokens', 'wasted_tokens', 'waste_ratio']]

In [ ]:
plot_df = df.copy()
plot_df['label'] = plot_df.apply(lambda r: 'naive' if pd.isna(r['block_size']) else f"block={int(r['block_size'])}", axis=1)
ax = plot_df.plot(kind='bar', x='label', y='waste_ratio', legend=False)
ax.set_title('Memory waste ratio: naive reservation vs paged blocks')
ax.set_ylabel('waste ratio')
ax.set_xlabel('allocation strategy')

The point is not that this toy simulation reproduces vLLM internals exactly. The point is conceptual: fixed max-length reservation wastes memory; block-based allocation reduces internal fragmentation.

In [ ]:
grid = make_memory_grid(requests[:32], block_size=128, max_rows=32)
plt.imshow(grid, aspect='auto')
plt.title('Toy block grid: used blocks vs unused reserved blocks')
plt.xlabel('block position')
plt.ylabel('request')
plt.colorbar(label='1=used, 0=unused')